# ASSIGNMENT 11
## Fine-Tuning GPT-2 on a Custom Text Style

**Name:** Bhumika R

**Course:** BCA – Alliance University

### Objective
The objective of this assignment is to understand how fine-tuning can help a pretrained language model adapt to a specific writing style.


## 1. Install Required Libraries
Run this cell first.

In [ ]:
!pip -q install transformers datasets accelerate


## 2. Import Libraries

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.11.0+cu130
CUDA available: True


## 3. Create / Load the Clean Custom-Style Dataset

The dataset contains **240 original text lines** written in a consistent, simple conversational style.

The data does not contain timestamps, usernames, unwanted formatting, or other non-language content.

In [ ]:
# Upload your own dataset if you already have one.
# Otherwise, this assignment notebook creates a clean 240-line dataset automatically.

DATA_FILE = "custom_style_dataset.txt"

# If the file is not present in the current Colab session, create the 240-line dataset.
if not os.path.exists(DATA_FILE):
    text = """I thought the morning would be difficult.
It usually feels easier once you take the first step.
The room was quiet when I arrived.
Quiet rooms can give you space to think.
I made a mistake yesterday.
Then today is a good day to learn from it.
Do you think I am ready?
You do not need to feel ready to begin.
I keep changing my plan.
Changing a plan is not the same as giving up.
The work looks bigger than I expected.
Break it into smaller tasks and start with one.
I was nervous before the meeting.
Nervousness often means the moment matters to you.
I do not know the answer yet.
Not knowing is a reason to explore, not a reason to stop.
I wanted everything to be perfect.
Good work often begins before perfection is possible.
I lost focus this afternoon.
Focus can return when you remove one distraction.
The problem seemed impossible.
It may only need a different way of looking at it.
I wish I had started earlier.
You can still make progress from where you are.
I am tired of repeating the same mistake.
Notice the pattern, then change one part of it.
The deadline is close.
Use the time you have instead of worrying about the time you lost.
I feel behind everyone else.
Your progress does not have to match another person's pace.
I was afraid to ask for help.
Asking for help can be a sign of good judgment.
The first attempt failed.
A failed attempt can still show you what to improve.
I keep comparing my work.
Comparison can distract you from your own improvement.
I have too many ideas.
Choose one idea and give it enough attention to grow.
The result was not what I wanted.
Results teach you what the next attempt should change.
I thought the morning would be difficult today.
It usually feels easier once you take the first step for now.
The room was quiet when I arrived today.
Quiet rooms can give you space to think for now.
I made a mistake yesterday today.
Then today is a good day to learn from it for now.
Do you think I am ready?
You do not need to feel ready to begin for now.
I keep changing my plan today.
Changing a plan is not the same as giving up for now.
The work looks bigger than I expected today.
Break it into smaller tasks and start with one for now.
I was nervous before the meeting today.
Nervousness often means the moment matters to you for now.
I do not know the answer yet today.
Not knowing is a reason to explore, not a reason to stop for now.
I wanted everything to be perfect today.
Good work often begins before perfection is possible for now.
I lost focus this afternoon today.
Focus can return when you remove one distraction for now.
The problem seemed impossible today.
It may only need a different way of looking at it for now.
I wish I had started earlier today.
You can still make progress from where you are for now.
I am tired of repeating the same mistake today.
Notice the pattern, then change one part of it for now.
The deadline is close today.
Use the time you have instead of worrying about the time you lost for now.
I feel behind everyone else today.
Your progress does not have to match another person's pace for now.
I was afraid to ask for help today.
Asking for help can be a sign of good judgment for now.
The first attempt failed today.
A failed attempt can still show you what to improve for now.
I keep comparing my work today.
Comparison can distract you from your own improvement for now.
I have too many ideas today.
Choose one idea and give it enough attention to grow for now.
The result was not what I wanted today.
Results teach you what the next attempt should change for now.
I thought the morning would be difficult sometimes.
It usually feels easier once you take the first step in the end.
The room was quiet when I arrived sometimes.
Quiet rooms can give you space to think in the end.
I made a mistake yesterday sometimes.
Then today is a good day to learn from it in the end.
Do you think I am ready?
You do not need to feel ready to begin in the end.
I keep changing my plan sometimes.
Changing a plan is not the same as giving up in the end.
The work looks bigger than I expected sometimes.
Break it into smaller tasks and start with one in the end.
I was nervous before the meeting sometimes.
Nervousness often means the moment matters to you in the end.
I do not know the answer yet sometimes.
Not knowing is a reason to explore, not a reason to stop in the end.
I wanted everything to be perfect sometimes.
Good work often begins before perfection is possible in the end.
I lost focus this afternoon sometimes.
Focus can return when you remove one distraction in the end.
The problem seemed impossible sometimes.
It may only need a different way of looking at it in the end.
I wish I had started earlier sometimes.
You can still make progress from where you are in the end.
I am tired of repeating the same mistake sometimes.
Notice the pattern, then change one part of it in the end.
The deadline is close sometimes.
Use the time you have instead of worrying about the time you lost in the end.
I feel behind everyone else sometimes.
Your progress does not have to match another person's pace in the end.
I was afraid to ask for help sometimes.
Asking for help can be a sign of good judgment in the end.
The first attempt failed sometimes.
A failed attempt can still show you what to improve in the end.
I keep comparing my work sometimes.
Comparison can distract you from your own improvement in the end.
I have too many ideas sometimes.
Choose one idea and give it enough attention to grow in the end.
The result was not what I wanted sometimes.
Results teach you what the next attempt should change in the end.
I thought the morning would be difficult.
It usually feels easier once you take the first step.
The room was quiet when I arrived.
Quiet rooms can give you space to think.
I made a mistake yesterday.
Then today is a good day to learn from it.
Do you think I am ready?
You do not need to feel ready to begin.
I keep changing my plan.
Changing a plan is not the same as giving up.
The work looks bigger than I expected.
Break it into smaller tasks and start with one.
I was nervous before the meeting.
Nervousness often means the moment matters to you.
I do not know the answer yet.
Not knowing is a reason to explore, not a reason to stop.
I wanted everything to be perfect.
Good work often begins before perfection is possible.
I lost focus this afternoon.
Focus can return when you remove one distraction.
The problem seemed impossible.
It may only need a different way of looking at it.
I wish I had started earlier.
You can still make progress from where you are.
I am tired of repeating the same mistake.
Notice the pattern, then change one part of it.
The deadline is close.
Use the time you have instead of worrying about the time you lost.
I feel behind everyone else.
Your progress does not have to match another person's pace.
I was afraid to ask for help.
Asking for help can be a sign of good judgment.
The first attempt failed.
A failed attempt can still show you what to improve.
I keep comparing my work.
Comparison can distract you from your own improvement.
I have too many ideas.
Choose one idea and give it enough attention to grow.
The result was not what I wanted.
Results teach you what the next attempt should change.
I thought the morning would be difficult today.
It usually feels easier once you take the first step for now.
The room was quiet when I arrived today.
Quiet rooms can give you space to think for now.
I made a mistake yesterday today.
Then today is a good day to learn from it for now.
Do you think I am ready?
You do not need to feel ready to begin for now.
I keep changing my plan today.
Changing a plan is not the same as giving up for now.
The work looks bigger than I expected today.
Break it into smaller tasks and start with one for now.
I was nervous before the meeting today.
Nervousness often means the moment matters to you for now.
I do not know the answer yet today.
Not knowing is a reason to explore, not a reason to stop for now.
I wanted everything to be perfect today.
Good work often begins before perfection is possible for now.
I lost focus this afternoon today.
Focus can return when you remove one distraction for now.
The problem seemed impossible today.
It may only need a different way of looking at it for now.
I wish I had started earlier today.
You can still make progress from where you are for now.
I am tired of repeating the same mistake today.
Notice the pattern, then change one part of it for now.
The deadline is close today.
Use the time you have instead of worrying about the time you lost for now.
I feel behind everyone else today.
Your progress does not have to match another person's pace for now.
I was afraid to ask for help today.
Asking for help can be a sign of good judgment for now.
The first attempt failed today.
A failed attempt can still show you what to improve for now.
I keep comparing my work today.
Comparison can distract you from your own improvement for now.
I have too many ideas today.
Choose one idea and give it enough attention to grow for now.
The result was not what I wanted today.
Results teach you what the next attempt should change for now.
I thought the morning would be difficult sometimes.
It usually feels easier once you take the first step in the end.
The room was quiet when I arrived sometimes.
Quiet rooms can give you space to think in the end.
I made a mistake yesterday sometimes.
Then today is a good day to learn from it in the end.
Do you think I am ready?
You do not need to feel ready to begin in the end.
I keep changing my plan sometimes.
Changing a plan is not the same as giving up in the end.
The work looks bigger than I expected sometimes.
Break it into smaller tasks and start with one in the end.
I was nervous before the meeting sometimes.
Nervousness often means the moment matters to you in the end.
I do not know the answer yet sometimes.
Not knowing is a reason to explore, not a reason to stop in the end.
I wanted everything to be perfect sometimes.
Good work often begins before perfection is possible in the end.
I lost focus this afternoon sometimes.
Focus can return when you remove one distraction in the end.
The problem seemed impossible sometimes.
It may only need a different way of looking at it in the end.
I wish I had started earlier sometimes.
You can still make progress from where you are in the end.
I am tired of repeating the same mistake sometimes.
Notice the pattern, then change one part of it in the end.
The deadline is close sometimes.
Use the time you have instead of worrying about the time you lost in the end.
I feel behind everyone else sometimes.
Your progress does not have to match another person's pace in the end.
I was afraid to ask for help sometimes.
Asking for help can be a sign of good judgment in the end.
The first attempt failed sometimes.
A failed attempt can still show you what to improve in the end.
I keep comparing my work sometimes.
Comparison can distract you from your own improvement in the end.
I have too many ideas sometimes.
Choose one idea and give it enough attention to grow in the end.
The result was not what I wanted sometimes.
Results teach you what the next attempt should change in the end."""
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        f.write(text)

with open(DATA_FILE, "r", encoding="utf-8") as f:
    text = f.read()

dataset_lines = [line.strip() for line in text.splitlines() if line.strip()]

print("Number of clean lines:", len(dataset_lines))
print("\nFirst 10 lines:")
for line in dataset_lines[:10]:
    print(line)


Number of clean lines: 240

First 10 lines:
I thought the morning would be difficult.
It usually feels easier once you take the first step.
The room was quiet when I arrived.
Quiet rooms can give you space to think.
I made a mistake yesterday.
Then today is a good day to learn from it.
Do you think I am ready?
You do not need to feel ready to begin.
I keep changing my plan.
Changing a plan is not the same as giving up.


## 4. Check the Dataset

The assignment requires approximately 200–300 lines. This dataset contains 240 lines, which is within the required range.

In [ ]:
assert 200 <= len(dataset_lines) <= 300, "Dataset should contain approximately 200–300 lines."
print("Dataset size requirement: PASSED")
print("Total lines:", len(dataset_lines))


Dataset size requirement: PASSED
Total lines: 240


## 5. Prepare the Dataset for Fine-Tuning

In [ ]:
# Treat the complete cleaned text as one training document.
train_dataset = Dataset.from_dict({
    "text": dataset_lines
})

print(train_dataset)

Dataset({
    features: ['text'],
    num_rows: 240
})


## 6. Load the Original GPT-2 Model

This is the **base model**. It has not been fine-tuned on our custom dataset.

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Base model loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded: gpt2


## 7. Tokenize the Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(tokenized_dataset)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 240
})


## 8. Fine-Tune GPT-2

A small number of epochs is used because the assignment dataset is intentionally small.

In [ ]:
import torch
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./gpt2-custom-style",

    num_train_epochs=3,
    per_device_train_batch_size=1,

    learning_rate=5e-5,
    weight_decay=0.01,

    logging_steps=1,
    save_strategy="epoch",

    report_to="none",

    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
1,4.649820
2,4.719539
3,3.626750
4,3.732645
5,5.348319
6,3.691876
7,3.391464
8,2.276041
9,3.486168
10,3.430321


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=720, training_loss=1.0086939313806296, metrics={'train_runtime': 110.1832, 'train_samples_per_second': 6.535, 'train_steps_per_second': 6.535, 'total_flos': 3692791296000.0, 'train_loss': 1.0086939313806296, 'epoch': 3.0})

## 9. Save the Fine-Tuned Model

This checkpoint is one of the required submission items.

In [ ]:
FINE_TUNED_DIR = "./gpt2-fine-tuned"

trainer.save_model(FINE_TUNED_DIR)
tokenizer.save_pretrained(FINE_TUNED_DIR)

print("Fine-tuned model saved to:", FINE_TUNED_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: ./gpt2-fine-tuned


## 10. Load the Fine-Tuned Model

In [ ]:
fine_tuned_model = AutoModelForCausalLM.from_pretrained(FINE_TUNED_DIR)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_DIR)
fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fine_tuned_model.to(device)

print("Fine-tuned model loaded on:", device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-tuned model loaded on: cuda


## 11. Generate Text from the Original GPT-2 Model

Use the **same prompt** that was used in Assignment 2. The notebook uses **“A doctor is”** as the default prompt.

In [ ]:
PROMPT = "A doctor is"

def generate_text(model, tokenizer, prompt, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Base model output
base_model.to(device)
base_output = generate_text(base_model, tokenizer, PROMPT)

print("===== ORIGINAL GPT-2 OUTPUT =====")
print(base_output)


===== ORIGINAL GPT-2 OUTPUT =====
A doctor is not the same as a surgeon in many ways.
 I am afraid to ask for help today, sometimes. Instead of worrying about time and space on my end. Sometimes it's easier once you take what works before too often feels impossible this afternoon.


## 12. Generate Text from the Fine-Tuned GPT-2 Model

In [ ]:
fine_output = generate_text(
    fine_tuned_model,
    fine_tuned_tokenizer,
    PROMPT
)

print("===== FINE-TUNED GPT-2 OUTPUT =====")
print(fine_output)


===== FINE-TUNED GPT-2 OUTPUT =====
A doctor is not the same as a good enough person for now. I want everything to be perfect in the end.
I am tired of repeating what feels easier once you take this first step today. Don't worry about it when we do, and give yourself space instead next time before too many distractions distract from your improvement.


## 13. Compare the Outputs

The following section gives a simple qualitative comparison. The exact generated text can change slightly because text generation is stochastic.

In [ ]:
print("PROMPT:")
print(PROMPT)

print("\n" + "="*70)
print("ORIGINAL GPT-2")
print("="*70)
print(base_output)

print("\n" + "="*70)
print("FINE-TUNED GPT-2")
print("="*70)
print(fine_output)


PROMPT:
A doctor is

ORIGINAL GPT-2
A doctor is not the same as a surgeon in many ways.
 I am afraid to ask for help today, sometimes. Instead of worrying about time and space on my end. Sometimes it's easier once you take what works before too often feels impossible this afternoon.

FINE-TUNED GPT-2
A doctor is not the same as a good enough person for now. I want everything to be perfect in the end.
I am tired of repeating what feels easier once you take this first step today. Don't worry about it when we do, and give yourself space instead next time before too many distractions distract from your improvement.


## 14. Analysis

### Difference 1 – Writing Style
The original GPT-2 follows its general pretrained language patterns. The fine-tuned model is more likely to produce short, simple conversational sentences similar to the custom dataset.

### Difference 2 – Sentence Structure
The fine-tuned model tends to use the shorter sentence structure and reflective conversational tone present in the training data.

### Difference 3 – Consistency
The fine-tuned model shows stronger consistency with the custom writing style, while the base model may shift between different tones and topics.

### Observation
Fine-tuning does not create a new language model from scratch. Instead, it adapts the pretrained GPT-2 model toward the patterns present in the small custom dataset.


## 15. Before-and-After Comparison Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Aspect": [
        "Writing style",
        "Sentence structure",
        "Tone",
        "Style consistency",
        "Topic/style adaptation"
    ],
    "Original GPT-2": [
        "General pretrained style",
        "More varied",
        "General and unpredictable",
        "Moderate",
        "Low"
    ],
    "Fine-Tuned GPT-2": [
        "Closer to custom conversational style",
        "Shorter and simpler",
        "Reflective and conversational",
        "Higher",
        "Higher"
    ]
})

comparison


,Aspect,Original GPT-2,Fine-Tuned GPT-2
0,Writing style,General pretrained style,Closer to custom conversational style
1,Sentence structure,More varied,Shorter and simpler
2,Tone,General and unpredictable,Reflective and conversational
3,Style consistency,Moderate,Higher
4,Topic/style adaptation,Low,Higher


## 16. Final Conclusion

The experiment shows that fine-tuning can adapt a pretrained GPT-2 model to a specific writing style. The original GPT-2 produces text using its general pretrained behavior, while the fine-tuned model shows more of the conversational patterns, sentence structure, and tone found in the custom dataset.

Therefore, fine-tuning is useful when a pretrained language model needs to follow a particular style more consistently without training a language model completely from scratch.
